In [1]:
# =============================================================================
# CELL 1: Imports and Configuration
# =============================================================================
# Run this cell first. If any import fails, install the missing package:
#   pip install sentence-transformers chromadb faster-whisper pyttsx3 PyPDF2 requests

# --- Standard library ---
import os
import json
import uuid
import tempfile
from pathlib import Path

# --- HTTP / API communication ---
import requests  # For talking to Ollama's REST API

# --- PDF text extraction ---
from PyPDF2 import PdfReader

# --- Text chunking ---
# We'll write our own chunker to avoid the heavy langchain dependency.
# If you later want langchain's RecursiveCharacterTextSplitter, swap it in.
import textwrap

# --- Embeddings ---
from sentence_transformers import SentenceTransformer

# --- Vector store ---
import chromadb
from chromadb.config import Settings

# --- Speech-to-Text ---
from faster_whisper import WhisperModel

# --- Text-to-Speech ---
import pyttsx3

# --- For audio playback in notebook (optional, for testing) ---
from IPython.display import Audio, display

# =============================================================================
# CONFIGURATION — All constants in one place for easy tuning
# =============================================================================

# -- Ollama --
OLLAMA_BASE_URL = "http://localhost:11434"  # Default Ollama address
OLLAMA_MODEL = "mistral"                    # Change to "llama3" or any model you've pulled

# -- Embedding model --
# all-MiniLM-L6-v2: 384 dimensions, ~80MB, great speed/quality tradeoff
# all-mpnet-base-v2: 768 dimensions, ~420MB, slightly better quality, slower
EMBEDDING_MODEL_NAME = "all-MiniLM-L6-v2"

# -- Chunking --
CHUNK_SIZE = 500         # Max characters per chunk
CHUNK_OVERLAP = 50       # Overlap between consecutive chunks (prevents losing context at boundaries)

# -- Retrieval --
TOP_K = 3                # Number of chunks to retrieve per query
CHROMA_COLLECTION = "rag_documents"  # Name of the ChromaDB collection
CHROMA_PERSIST_DIR = "./chroma_data" # Where ChromaDB stores vectors on disk

# -- Whisper STT --
# Model sizes: "tiny", "base", "small", "medium", "large-v3"
# "base" is a good starting point (~140MB, decent accuracy)
# Use "small" or "medium" if you have a GPU and want better accuracy
WHISPER_MODEL_SIZE = "base"

# -- Conversation history --
MAX_HISTORY_TURNS = 5    # Number of past Q&A pairs to include in the prompt

# -- File paths --
AUDIO_CACHE_DIR = "./audio_cache"
UPLOAD_DIR = "./uploads"

# Create directories if they don't exist
os.makedirs(AUDIO_CACHE_DIR, exist_ok=True)
os.makedirs(UPLOAD_DIR, exist_ok=True)
os.makedirs(CHROMA_PERSIST_DIR, exist_ok=True)

# =============================================================================
# VERIFICATION — Quick sanity checks
# =============================================================================
print("✓ All imports successful")
print(f"  Ollama URL:        {OLLAMA_BASE_URL}")
print(f"  LLM Model:         {OLLAMA_MODEL}")
print(f"  Embedding Model:   {EMBEDDING_MODEL_NAME}")
print(f"  Chunk size:        {CHUNK_SIZE} chars, {CHUNK_OVERLAP} overlap")
print(f"  Top-K retrieval:   {TOP_K}")
print(f"  Whisper model:     {WHISPER_MODEL_SIZE}")
print(f"  ChromaDB dir:      {CHROMA_PERSIST_DIR}")

c:\Users\HP\AppData\Local\Programs\Python\Python313\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


✓ All imports successful
  Ollama URL:        http://localhost:11434
  LLM Model:         mistral
  Embedding Model:   all-MiniLM-L6-v2
  Chunk size:        500 chars, 50 overlap
  Top-K retrieval:   3
  Whisper model:     base
  ChromaDB dir:      ./chroma_data


In [2]:
response = requests.get(f"{OLLAMA_BASE_URL}/api/tags")
print(response.json())

{'models': [{'name': 'mistral:latest', 'model': 'mistral:latest', 'modified_at': '2026-03-06T17:14:00.5563323-05:00', 'size': 4372824384, 'digest': '6577803aa9a036369e481d648a2baebb381ebc6e897f2bb9a766a2aa7bfbc1cf', 'details': {'parent_model': '', 'format': 'gguf', 'family': 'llama', 'families': ['llama'], 'parameter_size': '7.2B', 'quantization_level': 'Q4_K_M'}}]}


In [3]:
# =============================================================================
# CELL 2: Ollama Connection Test
# =============================================================================
# This cell verifies that:
#   1. Ollama is running and reachable
#   2. Your chosen model is loaded
#   3. You can send a prompt and receive a response
#
# Ollama exposes two main generation endpoints:
#   /api/generate  — single prompt in, single completion out
#   /api/chat      — conversation-style with a messages array (what we'll use in the RAG pipeline)
#
# We'll test both here so you see the difference.

# --- Test 1: Simple generation (single prompt, single response) ---
print("=" * 60)
print("TEST 1: /api/generate (single prompt)")
print("=" * 60)

response = requests.post(
    f"{OLLAMA_BASE_URL}/api/generate",
    json={
        "model": OLLAMA_MODEL,
        "prompt": "What is retrieval augmented generation? Answer in two sentences.",
        "stream": False  # Set to True if you want token-by-token streaming; False returns the full response at once
    }
)

# The response JSON contains several fields:
#   - "response": the generated text
#   - "done": whether generation is complete
#   - "total_duration": time taken in nanoseconds
#   - "eval_count": number of tokens generated
data = response.json()
print(f"Response: {data['response']}")
print(f"Tokens generated: {data.get('eval_count', 'N/A')}")
print(f"Time taken: {data.get('total_duration', 0) / 1e9:.2f} seconds")

# --- Test 2: Chat-style (conversation with message history) ---
# This is the format we'll use in the actual RAG pipeline because
# it naturally supports multi-turn conversation history.
print("\n" + "=" * 60)
print("TEST 2: /api/chat (conversation style)")
print("=" * 60)

response = requests.post(
    f"{OLLAMA_BASE_URL}/api/chat",
    json={
        "model": OLLAMA_MODEL,
        "messages": [
            # System message sets the LLM's behavior/persona
            {
                "role": "system",
                "content": "You are a helpful assistant. Keep your answers brief and clear."
            },
            # User message is the actual query
            {
                "role": "user",
                "content": "Explain what an embedding is in one sentence."
            }
        ],
        "stream": False
    }
)

data = response.json()
# In chat mode, the response is nested under "message.content"
print(f"Response: {data['message']['content']}")
print(f"Tokens generated: {data.get('eval_count', 'N/A')}")
print(f"Time taken: {data.get('total_duration', 0) / 1e9:.2f} seconds")

# --- Test 3: Verify the format we'll use for RAG ---
# In our RAG pipeline, the prompt will look like this:
# System: You are an assistant that answers based on provided context.
# User: [context chunks + question]
# This test simulates that pattern.
print("\n" + "=" * 60)
print("TEST 3: Simulated RAG prompt format")
print("=" * 60)

fake_context = """
Company Policy Document:
Employees are entitled to 21 days of paid leave per calendar year.
Unused leave cannot be carried over to the next year.
Leave requests must be submitted at least two weeks in advance.
"""

user_question = "How many vacation days do I get?"

response = requests.post(
    f"{OLLAMA_BASE_URL}/api/chat",
    json={
        "model": OLLAMA_MODEL,
        "messages": [
            {
                "role": "system",
                "content": (
                    "You are a helpful assistant. Answer the user's question based ONLY on the "
                    "provided context. If the context doesn't contain the answer, say so. "
                    "Do not make up information."
                )
            },
            {
                "role": "user",
                "content": f"Context:\n{fake_context}\n\nQuestion: {user_question}"
            }
        ],
        "stream": False
    }
)

data = response.json()
print(f"Question: {user_question}")
print(f"Response: {data['message']['content']}")

print("\n✓ Ollama connection verified — all three tests passed")

TEST 1: /api/generate (single prompt)
Response:  Retrieval-augmented generation is a method in artificial intelligence where a model leverages external data sources, often a large corpus of text, to improve its responses by incorporating relevant information from the data sources during the generation process. This approach aims to enhance the model's performance and provide more accurate and contextually relevant responses.
Tokens generated: 70
Time taken: 39.83 seconds

TEST 2: /api/chat (conversation style)
Response:  An embedding is a way of representing data in a different space, usually as a low-dimensional vector, to capture complex relationships and facilitate computations.
Tokens generated: 31
Time taken: 0.74 seconds

TEST 3: Simulated RAG prompt format
Question: How many vacation days do I get?
Response:  According to the provided context, you are entitled to 21 days of paid leave per calendar year.

✓ Ollama connection verified — all three tests passed


In [4]:
# =============================================================================
# CELL 3: Document Loading and Text Extraction
# =============================================================================
# This cell handles the first step of the ingestion pipeline:
#   Raw file (PDF or TXT) → Extracted plain text
#
# We support two formats:
#   - PDF: Using PyPDF2 to extract text page by page
#   - TXT: Simple file read
#
# PyPDF2 works well for text-based PDFs. If your PDFs are scanned images
# (no selectable text), you'd need OCR (e.g., pytesseract) — but that's
# a different problem we won't tackle here.

def extract_text_from_pdf(file_path: str) -> str:
    """
    Extract all text from a PDF file, page by page.
    
    Args:
        file_path: Path to the PDF file
        
    Returns:
        The full extracted text as a single string
    """
    reader = PdfReader(file_path)
    pages_text = []
    
    for i, page in enumerate(reader.pages):
        text = page.extract_text()
        if text:  # Some pages might be images or empty
            pages_text.append(text)
        else:
            print(f"  ⚠ Page {i+1}: No text extracted (might be an image or blank)")
    
    # Join all pages with double newline to preserve page boundaries
    return "\n\n".join(pages_text)


def extract_text_from_txt(file_path: str) -> str:
    """
    Read a plain text file.
    
    Args:
        file_path: Path to the text file
        
    Returns:
        The file contents as a string
    """
    with open(file_path, "r", encoding="utf-8") as f:
        return f.read()


def load_document(file_path: str) -> str:
    """
    Load a document and extract its text content.
    Dispatches to the right extractor based on file extension.
    
    Args:
        file_path: Path to the document (PDF or TXT)
        
    Returns:
        Extracted text content
        
    Raises:
        ValueError: If the file format is not supported
    """
    path = Path(file_path)
    
    if not path.exists():
        raise FileNotFoundError(f"File not found: {file_path}")
    
    ext = path.suffix.lower()
    
    if ext == ".pdf":
        print(f"Loading PDF: {path.name}")
        text = extract_text_from_pdf(file_path)
    elif ext == ".txt":
        print(f"Loading TXT: {path.name}")
        text = extract_text_from_txt(file_path)
    else:
        raise ValueError(f"Unsupported file format: {ext}. Use .pdf or .txt")
    
    return text


# =============================================================================
# TEST: Load your sample document
# =============================================================================
# Place your file path here — adjust if your file is in a different location
SAMPLE_DOC_PATH = "./sample_doc.pdf"

raw_text = load_document(SAMPLE_DOC_PATH)

# --- Print stats and a preview ---
print(f"\n✓ Text extracted successfully")
print(f"  Total characters: {len(raw_text):,}")
print(f"  Approx tokens:    ~{len(raw_text) // 4:,} (rough estimate: 1 token ≈ 4 chars)")
print(f"  Total pages:      {len(PdfReader(SAMPLE_DOC_PATH).pages)}")

print(f"\n{'=' * 60}")
print("PREVIEW (first 1000 characters):")
print("=" * 60)
print(raw_text[:1000])
print("..." if len(raw_text) > 1000 else "")

Loading PDF: sample_doc.pdf

✓ Text extracted successfully
  Total characters: 60,719
  Approx tokens:    ~15,179 (rough estimate: 1 token ≈ 4 chars)
  Total pages:      46

PREVIEW (first 1000 characters):
 
1 
 Distributed Emotion Detection System  
Major Project submitted to  
Amity School of Engineering and Technology, Amity 
University  
In partial fulfillment for the requirements for the degree  of 
 
I. [B. Tech + M. Tech] in Artificial Intelligence and Robotics  
From  
Department of Artificial Intelligence, Amity School of 
Engineering and Technology  
 
By 
 
Angshuman Basu (A910112520006)  
 
Under the Guidance of  
Prof. Pronaya Bhattacharya  (309881 ) 
 
 
Amity School of Engineering and Technology  
Amity University  
Kolkata – 700135  
West Bengal, India  
    3rd December  2024  
  


 
2 
 Amity School of Engineering and Technology  
Amity University  
 
 
 
Certificate  
 
This is to certify that the project work entitled “ Distributed Emotion Detection 
System ” has 

In [5]:
# =============================================================================
# CELL 4: Text Chunking
# =============================================================================
# This cell splits the raw extracted text into overlapping chunks.
#
# Why chunk?
#   - Embedding models work best on shorter passages (a few sentences)
#   - The LLM's context window is limited — we want to send only relevant pieces
#   - Smaller chunks = more precise retrieval (less noise per chunk)
#
# Why overlap?
#   - Important information might fall right at a chunk boundary
#   - Overlap ensures that context isn't lost between consecutive chunks
#   - Example: if a sentence spans characters 490-520 and your chunk size is 500,
#     without overlap you'd split that sentence across two chunks
#
# Our approach: split on sentence boundaries when possible, fall back to
# character-level splitting only when a single sentence exceeds chunk_size.
# This is cleaner than naive fixed-length slicing which can cut mid-word.

def chunk_text(text: str, chunk_size: int = CHUNK_SIZE, overlap: int = CHUNK_OVERLAP) -> list[str]:
    """
    Split text into overlapping chunks, respecting sentence boundaries.
    
    Strategy:
        1. Split text into sentences (on '. ', '? ', '! ', and newlines)
        2. Accumulate sentences into a chunk until adding the next sentence
           would exceed chunk_size
        3. Save that chunk, then start the next chunk with overlap by
           rewinding a few sentences
    
    Args:
        text:       The full document text
        chunk_size: Maximum characters per chunk
        overlap:    Approximate overlap in characters between consecutive chunks
        
    Returns:
        List of text chunks
    """
    # -- Step 1: Split into sentences --
    # We split on common sentence endings followed by a space or newline.
    # This is a simple heuristic — not perfect for abbreviations like "Dr."
    # but good enough for most documents.
    import re
    sentences = re.split(r'(?<=[.!?])\s+|\n\n+', text)
    
    # Clean up: remove empty strings and strip whitespace
    sentences = [s.strip() for s in sentences if s.strip()]
    
    # -- Step 2: Accumulate sentences into chunks --
    chunks = []
    current_chunk = []       # List of sentences in the current chunk
    current_length = 0       # Character count of the current chunk
    
    for sentence in sentences:
        sentence_length = len(sentence)
        
        # If a single sentence exceeds chunk_size, we have to include it as-is
        # (splitting mid-sentence would be worse)
        if sentence_length > chunk_size:
            # Save current chunk if it has content
            if current_chunk:
                chunks.append(" ".join(current_chunk))
            # Add the oversized sentence as its own chunk
            chunks.append(sentence)
            current_chunk = []
            current_length = 0
            continue
        
        # Would adding this sentence exceed the chunk size?
        if current_length + sentence_length + 1 > chunk_size:  # +1 for the space
            # Save the current chunk
            chunks.append(" ".join(current_chunk))
            
            # -- Step 3: Create overlap --
            # Walk backwards through current_chunk's sentences until we've
            # accumulated approximately 'overlap' characters worth of text
            overlap_sentences = []
            overlap_length = 0
            for s in reversed(current_chunk):
                if overlap_length + len(s) > overlap:
                    break
                overlap_sentences.insert(0, s)
                overlap_length += len(s) + 1  # +1 for space
            
            # Start new chunk with the overlap sentences
            current_chunk = overlap_sentences
            current_length = overlap_length
        
        # Add the sentence to the current chunk
        current_chunk.append(sentence)
        current_length += sentence_length + 1  # +1 for space
    
    # Don't forget the last chunk
    if current_chunk:
        chunks.append(" ".join(current_chunk))
    
    return chunks


# =============================================================================
# TEST: Chunk the extracted text
# =============================================================================
chunks = chunk_text(raw_text)

print(f"✓ Chunking complete")
print(f"  Total chunks:     {len(chunks)}")
print(f"  Chunk size range: {min(len(c) for c in chunks)} - {max(len(c) for c in chunks)} characters")
print(f"  Average size:     {sum(len(c) for c in chunks) // len(chunks)} characters")

# -- Show first 3 chunks so you can verify quality --
for i, chunk in enumerate(chunks[:3]):
    print(f"\n{'=' * 60}")
    print(f"CHUNK {i+1} ({len(chunk)} chars):")
    print("=" * 60)
    print(chunk)

# -- Verify overlap is working --
# Check that consecutive chunks share some text
if len(chunks) >= 2:
    print(f"\n{'=' * 60}")
    print("OVERLAP CHECK:")
    print("=" * 60)
    # Find common text between chunk 1 and chunk 2
    end_of_first = chunks[0][-100:]    # Last 100 chars of chunk 1
    start_of_second = chunks[1][:100]  # First 100 chars of chunk 2
    print(f"End of Chunk 1:    ...{end_of_first}")
    print(f"Start of Chunk 2:  {start_of_second}...")
    print(f"\n(You should see some overlapping text between these two)")

✓ Chunking complete
  Total chunks:     122
  Chunk size range: 179 - 1655 characters
  Average size:     491 characters

CHUNK 1 (431 chars):
1 
 Distributed Emotion Detection System  
Major Project submitted to  
Amity School of Engineering and Technology, Amity 
University  
In partial fulfillment for the requirements for the degree  of 
 
I. [B. Tech + M. Tech] in Artificial Intelligence and Robotics  
From  
Department of Artificial Intelligence, Amity School of 
Engineering and Technology  
 
By 
 
Angshuman Basu (A910112520006)  
 
Under the Guidance of  
Prof.

CHUNK 2 (417 chars):
Pronaya Bhattacharya  (309881 ) 
 
 
Amity School of Engineering and Technology  
Amity University  
Kolkata – 700135  
West Bengal, India  
    3rd December  2024 2 
 Amity School of Engineering and Technology  
Amity University  
 
 
 
Certificate  
 
This is to certify that the project work entitled “ Distributed Emotion Detection 
System ” has been prepared by Angshuman Basu  under my supervision

In [6]:
# =============================================================================
# CELL 5: Embedding Model Setup and Test
# =============================================================================
# This cell loads the sentence-transformers embedding model and verifies
# it produces vectors of the expected dimensionality.
#
# The embedding model is separate from the LLM (Ollama/Mistral).
#   - Embedding model: converts text → fixed-size vector (for similarity search)
#   - LLM: converts prompt → generated text (for answering questions)
#
# all-MiniLM-L6-v2:
#   - 384 dimensions
#   - ~80MB download (first run only, then cached)
#   - Trained specifically for semantic similarity tasks
#   - Fast on CPU — can embed hundreds of sentences per second

# --- Load the model (downloads on first run, cached after that) ---
print(f"Loading embedding model: {EMBEDDING_MODEL_NAME}")
embedding_model = SentenceTransformer(EMBEDDING_MODEL_NAME)
print(f"✓ Model loaded\n")

# =============================================================================
# TEST 1: Embed a single sentence and inspect the output
# =============================================================================
test_sentence = "Employees are entitled to 21 days of paid leave annually."
test_embedding = embedding_model.encode(test_sentence)

print(f"Input:       \"{test_sentence}\"")
print(f"Vector shape: {test_embedding.shape}")       # Should be (384,)
print(f"Vector dtype:  {test_embedding.dtype}")       # Should be float32
print(f"First 10 values: {test_embedding[:10]}")      # Just to see what the numbers look like
print(f"Value range:  [{test_embedding.min():.4f}, {test_embedding.max():.4f}]")

# =============================================================================
# TEST 2: Verify semantic similarity works
# =============================================================================
# We'll embed three sentences and check that semantically similar ones
# have higher cosine similarity than dissimilar ones.
from numpy import dot
from numpy.linalg import norm

def cosine_similarity(a, b):
    """Compute cosine similarity between two vectors."""
    return dot(a, b) / (norm(a) * norm(b))

sentences = [
    "How much vacation do I get?",              # Query about leave
    "Employees receive 21 days of paid leave.",  # Semantically similar to query
    "The company was founded in 2003.",          # Semantically different
]

embeddings = embedding_model.encode(sentences)

print(f"\n{'=' * 60}")
print("SEMANTIC SIMILARITY TEST")
print("=" * 60)
print(f"Sentence A: \"{sentences[0]}\"")
print(f"Sentence B: \"{sentences[1]}\"")
print(f"Sentence C: \"{sentences[2]}\"")

sim_ab = cosine_similarity(embeddings[0], embeddings[1])
sim_ac = cosine_similarity(embeddings[0], embeddings[2])

print(f"\nSimilarity (A vs B): {sim_ab:.4f}  ← should be HIGH (same topic)")
print(f"Similarity (A vs C): {sim_ac:.4f}  ← should be LOW (different topic)")
print(f"\nDifference: {sim_ab - sim_ac:.4f}")

if sim_ab > sim_ac:
    print("✓ Semantic similarity is working correctly — similar sentences score higher")
else:
    print("✗ Something is off — similar sentences should score higher")

# =============================================================================
# TEST 3: Embed a few actual chunks from your document
# =============================================================================
# This confirms the model handles your real document content properly.
print(f"\n{'=' * 60}")
print("EMBEDDING YOUR DOCUMENT CHUNKS")
print("=" * 60)

# Embed first 5 chunks as a quick test
sample_chunks = chunks[:5]
chunk_embeddings = embedding_model.encode(sample_chunks)

print(f"Embedded {len(sample_chunks)} chunks")
print(f"Embeddings shape: {chunk_embeddings.shape}")  # Should be (5, 384)
print(f"\n✓ Embedding model is ready for full ingestion in Cell 6")

Loading embedding model: all-MiniLM-L6-v2


c:\Users\HP\AppData\Local\Programs\Python\Python313\Lib\site-packages\huggingface_hub\file_download.py:129: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\HP\.cache\huggingface\hub\models--sentence-transformers--all-MiniLM-L6-v2. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)
Loading weights: 100%|██████████| 103/103 [00:00<00:00, 1236.33it/s, Materializin

✓ Model loaded

Input:       "Employees are entitled to 21 days of paid leave annually."
Vector shape: (384,)
Vector dtype:  float32
First 10 values: [ 0.06953927  0.04727192  0.08735855  0.05954438 -0.02603118  0.09524038
 -0.04651671 -0.06369214 -0.03951992 -0.04777099]
Value range:  [-0.1666, 0.1525]

SEMANTIC SIMILARITY TEST
Sentence A: "How much vacation do I get?"
Sentence B: "Employees receive 21 days of paid leave."
Sentence C: "The company was founded in 2003."

Similarity (A vs B): 0.4008  ← should be HIGH (same topic)
Similarity (A vs C): -0.0001  ← should be LOW (different topic)

Difference: 0.4009
✓ Semantic similarity is working correctly — similar sentences score higher

EMBEDDING YOUR DOCUMENT CHUNKS
Embedded 5 chunks
Embeddings shape: (5, 384)

✓ Embedding model is ready for full ingestion in Cell 6


In [7]:
# =============================================================================
# CELL 6: ChromaDB Setup and Document Ingestion
# =============================================================================
# This cell:
#   1. Initializes ChromaDB with persistent storage (survives restarts)
#   2. Creates a collection (think of it as a "table" for vectors)
#   3. Embeds ALL chunks from your document
#   4. Stores the embeddings + original text + metadata in ChromaDB
#
# ChromaDB stores three things per entry:
#   - id:        A unique identifier for each chunk
#   - embedding: The 384-dim vector (used for similarity search)
#   - document:  The original chunk text (returned with search results)
#   - metadata:  Optional key-value pairs (we'll store chunk index and source filename)

# --- Step 1: Initialize ChromaDB client with persistence ---
# PersistentClient saves to disk so your vectors survive kernel restarts.
# If you used Client() instead, everything would be in-memory only.
chroma_client = chromadb.PersistentClient(path=CHROMA_PERSIST_DIR)

# --- Step 2: Create (or get existing) collection ---
# get_or_create_collection: if it already exists, returns it; otherwise creates it.
# This makes the cell safely re-runnable.
#
# We pass our embedding function as None because we'll provide pre-computed
# embeddings ourselves. This keeps us in control of which embedding model is used.
collection = chroma_client.get_or_create_collection(
    name=CHROMA_COLLECTION,
    metadata={"hnsw:space": "cosine"}  # Use cosine similarity for search (matches our earlier tests)
)

# --- Step 3: Check if collection already has data ---
# If you re-run this cell, we don't want to duplicate entries.
existing_count = collection.count()
if existing_count > 0:
    print(f"⚠ Collection '{CHROMA_COLLECTION}' already has {existing_count} entries.")
    print(f"  Clearing collection to re-ingest fresh...")
    chroma_client.delete_collection(name=CHROMA_COLLECTION)
    collection = chroma_client.create_collection(
        name=CHROMA_COLLECTION,
        metadata={"hnsw:space": "cosine"}
    )
    print(f"  ✓ Collection cleared\n")

# --- Step 4: Embed all chunks ---
print(f"Embedding {len(chunks)} chunks...")
all_embeddings = embedding_model.encode(chunks, show_progress_bar=True)
print(f"✓ Embeddings generated — shape: {all_embeddings.shape}")

# --- Step 5: Store in ChromaDB ---
# ChromaDB expects:
#   ids:        list of unique string IDs
#   embeddings: list of vectors (as lists, not numpy arrays)
#   documents:  list of original text strings
#   metadatas:  list of metadata dicts

# Generate unique IDs for each chunk
ids = [f"chunk_{i}" for i in range(len(chunks))]

# Build metadata for each chunk
# Storing the source file and chunk index makes it easier to trace
# which part of which document a result came from
source_filename = Path(SAMPLE_DOC_PATH).name
metadatas = [
    {
        "source": source_filename,
        "chunk_index": i,
        "char_count": len(chunk)
    }
    for i, chunk in enumerate(chunks)
]

# ChromaDB has a batch size limit, so we insert in batches of 100
BATCH_SIZE = 100
for start in range(0, len(chunks), BATCH_SIZE):
    end = min(start + BATCH_SIZE, len(chunks))
    collection.add(
        ids=ids[start:end],
        embeddings=all_embeddings[start:end].tolist(),  # Convert numpy to list
        documents=chunks[start:end],
        metadatas=metadatas[start:end]
    )
    print(f"  Stored batch {start}-{end}")

# --- Step 6: Verify ---
final_count = collection.count()
print(f"\n✓ Ingestion complete")
print(f"  Collection:    {CHROMA_COLLECTION}")
print(f"  Total entries: {final_count}")
print(f"  Storage:       {CHROMA_PERSIST_DIR}")

# Quick sanity check: peek at one entry
sample = collection.peek(limit=1)
print(f"\n{'=' * 60}")
print("SAMPLE ENTRY (first stored chunk):")
print("=" * 60)
print(f"  ID:       {sample['ids'][0]}")
print(f"  Metadata: {sample['metadatas'][0]}")
print(f"  Text:     {sample['documents'][0][:200]}...")

Embedding 122 chunks...


Batches: 100%|██████████| 4/4 [00:00<00:00,  8.20it/s]

✓ Embeddings generated — shape: (122, 384)
  Stored batch 0-100
  Stored batch 100-122

✓ Ingestion complete
  Collection:    rag_documents
  Total entries: 122
  Storage:       ./chroma_data

SAMPLE ENTRY (first stored chunk):
  ID:       chunk_0
  Metadata: {'char_count': 431, 'chunk_index': 0, 'source': 'sample_doc.pdf'}
  Text:     1 
 Distributed Emotion Detection System  
Major Project submitted to  
Amity School of Engineering and Technology, Amity 
University  
In partial fulfillment for the requirements for the degree  of 
...


In [8]:
# =============================================================================
# CELL 7: Retrieval Function and Testing
# =============================================================================
# This is the core of RAG — given a user query, find the most relevant
# chunks from ChromaDB.
#
# The flow:
#   1. Embed the query using the same model that embedded the chunks
#   2. Send the query embedding to ChromaDB
#   3. ChromaDB computes cosine similarity against all stored vectors
#   4. Return the top-k most similar chunks with their scores
#
# IMPORTANT: The query MUST be embedded with the same model as the chunks.
# If you used a different model, the vector spaces wouldn't align and
# similarity scores would be meaningless.

def retrieve(query: str, top_k: int = TOP_K) -> dict:
    """
    Retrieve the most relevant chunks for a given query.
    
    Args:
        query:  The user's question as a string
        top_k:  Number of chunks to retrieve
        
    Returns:
        dict with keys:
            - documents:  list of chunk texts
            - metadatas:  list of metadata dicts
            - distances:  list of similarity distances (lower = more similar for cosine)
            - ids:        list of chunk IDs
    """
    # Step 1: Embed the query
    query_embedding = embedding_model.encode(query).tolist()
    
    # Step 2: Query ChromaDB
    # ChromaDB returns "distances" not "similarities" — for cosine space:
    #   distance = 1 - cosine_similarity
    #   So distance 0.0 = identical, distance 2.0 = opposite
    #   Lower distance = more relevant
    results = collection.query(
        query_embeddings=[query_embedding],
        n_results=top_k,
        include=["documents", "metadatas", "distances"]
    )
    
    # Results come wrapped in an extra list (because you can batch queries),
    # so we unwrap the first (and only) query's results
    return {
        "documents": results["documents"][0],
        "metadatas": results["metadatas"][0],
        "distances": results["distances"][0],
        "ids": results["ids"][0]
    }


def format_context(retrieved: dict) -> str:
    """
    Format retrieved chunks into a context string for the LLM prompt.
    
    Args:
        retrieved: Output from retrieve()
        
    Returns:
        Formatted context string with chunk separators
    """
    context_parts = []
    for i, (doc, meta, dist) in enumerate(zip(
        retrieved["documents"],
        retrieved["metadatas"],
        retrieved["distances"]
    )):
        # Convert distance to similarity for readability
        similarity = 1 - dist
        context_parts.append(
            f"[Source: {meta['source']}, Chunk {meta['chunk_index']}, "
            f"Relevance: {similarity:.2%}]\n{doc}"
        )
    
    return "\n\n---\n\n".join(context_parts)


# =============================================================================
# TEST: Try several queries against your document
# =============================================================================
test_queries = [
    "What is this project about?",
    "Who is the author?",
    "What technologies or tools were used?",
]

for query in test_queries:
    print(f"{'=' * 60}")
    print(f"QUERY: \"{query}\"")
    print(f"{'=' * 60}")
    
    results = retrieve(query, top_k=3)
    
    for i, (doc, meta, dist) in enumerate(zip(
        results["documents"],
        results["metadatas"],
        results["distances"]
    )):
        similarity = 1 - dist
        print(f"\n  --- Result {i+1} (Relevance: {similarity:.2%}) ---")
        print(f"  Chunk ID:    {meta['chunk_index']}")
        print(f"  Source:      {meta['source']}")
        print(f"  Text:        {doc[:300]}{'...' if len(doc) > 300 else ''}")
    
    print()

# =============================================================================
# TEST: View the formatted context that will be sent to the LLM
# =============================================================================
print("=" * 60)
print("FORMATTED CONTEXT (as the LLM will see it):")
print("=" * 60)
sample_results = retrieve("What is this project about?")
formatted = format_context(sample_results)
print(formatted)

QUERY: "What is this project about?"

  --- Result 1 (Relevance: 35.61%) ---
  Chunk ID:    2
  Source:      sample_doc.pdf
  Text:        The thesis is his original work completed after careful research and investigation. The 
work of the thesis is of the standard expected of a candidate for I [B. Tech + M. Tech] 
(Artificial Intelligence and Robotics) , and I recommend that it be sent for evaluation. Date: 3rd December  2024  
      ...

  --- Result 2 (Relevance: 35.55%) ---
  Chunk ID:    6
  Source:      sample_doc.pdf
  Text:        Signature of Student 4 
 DEDICATION  
 
This project report is dedicated to researchers, technologists, and mental health 
professionals who strive to better understand and address the complexities of 
human emotions. It honors the efforts of those working to bridge the gap between 
artificial intel...

  --- Result 3 (Relevance: 34.49%) ---
  Chunk ID:    1
  Source:      sample_doc.pdf
  Text:        Pronaya Bhattacharya  (309881 ) 
 
 
Amity School

In [9]:
# =============================================================================
# CELL 7b: Improved Retrieval — Fix for noisy front-matter
# =============================================================================
# Two changes:
#   1. Increase TOP_K from 3 to 5 (cast a wider net)
#   2. Re-ingest with a chunk quality filter to remove low-content chunks
#
# The quality filter skips chunks that are mostly formatting noise:
#   - Too few actual words (< 20 words)
#   - High ratio of special characters to letters

import re

def is_quality_chunk(chunk: str, min_words: int = 20) -> bool:
    """
    Filter out low-content chunks (title pages, signature lines, etc.)
    
    Args:
        chunk:     The chunk text
        min_words: Minimum word count to be considered useful
        
    Returns:
        True if the chunk has enough meaningful content
    """
    # Count actual words (not symbols, not single characters)
    words = [w for w in chunk.split() if len(w) > 1 and any(c.isalpha() for c in w)]
    
    if len(words) < min_words:
        return False
    
    # Calculate ratio of letters to total characters
    letters = sum(1 for c in chunk if c.isalpha())
    total = len(chunk)
    if total == 0:
        return False
    
    letter_ratio = letters / total
    
    # If less than 40% of characters are letters, it's likely
    # formatting noise (dots, dashes, signatures, etc.)
    if letter_ratio < 0.40:
        return False
    
    return True


# --- Re-chunk and filter ---
print("Re-filtering chunks for quality...\n")

original_count = len(chunks)
filtered_chunks = [c for c in chunks if is_quality_chunk(c)]
removed_count = original_count - len(filtered_chunks)

print(f"  Original chunks:  {original_count}")
print(f"  Removed (noise):  {removed_count}")
print(f"  Remaining:        {len(filtered_chunks)}")

# Show a few removed chunks so you can verify the filter is correct
removed_chunks = [c for c in chunks if not is_quality_chunk(c)]
if removed_chunks:
    print(f"\n  Examples of REMOVED chunks:")
    for i, rc in enumerate(removed_chunks[:3]):
        preview = rc[:150].replace('\n', ' ')
        print(f"    [{i+1}] \"{preview}...\"")

# --- Re-ingest filtered chunks into ChromaDB ---
print(f"\nRe-ingesting filtered chunks into ChromaDB...")

# Clear old collection
chroma_client.delete_collection(name=CHROMA_COLLECTION)
collection = chroma_client.create_collection(
    name=CHROMA_COLLECTION,
    metadata={"hnsw:space": "cosine"}
)

# Embed filtered chunks
all_embeddings = embedding_model.encode(filtered_chunks, show_progress_bar=True)

# Store with updated IDs and metadata
ids = [f"chunk_{i}" for i in range(len(filtered_chunks))]
source_filename = Path(SAMPLE_DOC_PATH).name
metadatas = [
    {"source": source_filename, "chunk_index": i, "char_count": len(c)}
    for i, c in enumerate(filtered_chunks)
]

BATCH_SIZE = 100
for start in range(0, len(filtered_chunks), BATCH_SIZE):
    end = min(start + BATCH_SIZE, len(filtered_chunks))
    collection.add(
        ids=ids[start:end],
        embeddings=all_embeddings[start:end].tolist(),
        documents=filtered_chunks[start:end],
        metadatas=metadatas[start:end]
    )

print(f"✓ Re-ingestion complete — {collection.count()} chunks stored\n")

# --- Update TOP_K ---
TOP_K = 5
print(f"TOP_K updated to {TOP_K}\n")

# =============================================================================
# RE-TEST: Same queries, should see better results now
# =============================================================================
test_queries = [
    "What is this project about?",
    "Who is the author?",
    "What technologies or tools were used?",
]

for query in test_queries:
    print(f"{'=' * 60}")
    print(f"QUERY: \"{query}\"")
    print(f"{'=' * 60}")
    
    results = retrieve(query, top_k=TOP_K)
    
    for i, (doc, meta, dist) in enumerate(zip(
        results["documents"],
        results["metadatas"],
        results["distances"]
    )):
        similarity = 1 - dist
        print(f"\n  --- Result {i+1} (Relevance: {similarity:.2%}) ---")
        print(f"  Chunk ID:    {meta['chunk_index']}")
        print(f"  Text:        {doc[:300]}{'...' if len(doc) > 300 else ''}")
    
    print()

Re-filtering chunks for quality...

  Original chunks:  122
  Removed (noise):  5
  Remaining:        117

  Examples of REMOVED chunks:
    [1] "Date: 3rd December  2024                              ……………………………... Signature of Student                              ……………………………... Signature of Stu..."
    [2] "41   Input Templa te:    <!DOCTYPE html>   <html lang="en">   <head>       <meta charset="UTF -8">      <meta name="viewport" content="width=device -w..."
    [3] "43   Output Templa te:    <!DOCTYPE html>   <html lang="en">   <head>       <meta charset="UTF -8">      <meta name="viewport" content="width=device -..."

Re-ingesting filtered chunks into ChromaDB...


Batches: 100%|██████████| 4/4 [00:00<00:00, 10.21it/s]

✓ Re-ingestion complete — 117 chunks stored

TOP_K updated to 5

QUERY: "What is this project about?"

  --- Result 1 (Relevance: 35.61%) ---
  Chunk ID:    2
  Text:        The thesis is his original work completed after careful research and investigation. The 
work of the thesis is of the standard expected of a candidate for I [B. Tech + M. Tech] 
(Artificial Intelligence and Robotics) , and I recommend that it be sent for evaluation. Date: 3rd December  2024  
      ...

  --- Result 2 (Relevance: 35.55%) ---
  Chunk ID:    5
  Text:        Signature of Student 4 
 DEDICATION  
 
This project report is dedicated to researchers, technologists, and mental health 
professionals who strive to better understand and address the complexities of 
human emotions. It honors the efforts of those working to bridge the gap between 
artificial intel...

  --- Result 3 (Relevance: 34.49%) ---
  Chunk ID:    1
  Text:        Pronaya Bhattacharya  (309881 ) 
 
 
Amity School of Engineering and Techn

In [10]:
# =============================================================================
# CELL 7c: Testing with more specific queries
# =============================================================================
# Vague queries like "What is this project about?" match loosely with
# everything, which is why front-matter scores almost the same as real content.
# More specific queries give the embedding model a clearer semantic target.

specific_queries = [
    "How does the system detect emotions from text or images?",
    "What deep learning model architecture is used for emotion detection?",
    "How is Docker or Kubernetes used in the system?",
    "What datasets were used to train the emotion detection model?",
    "How does the blockchain component work in the system?",
]

for query in specific_queries:
    print(f"{'=' * 60}")
    print(f"QUERY: \"{query}\"")
    print(f"{'=' * 60}")
    
    results = retrieve(query, top_k=3)  # Top 3 is enough to see the pattern
    
    for i, (doc, meta, dist) in enumerate(zip(
        results["documents"],
        results["metadatas"],
        results["distances"]
    )):
        similarity = 1 - dist
        print(f"\n  --- Result {i+1} (Relevance: {similarity:.2%}) ---")
        print(f"  Chunk ID:    {meta['chunk_index']}")
        print(f"  Text:        {doc[:300]}{'...' if len(doc) > 300 else ''}")
    
    print()

QUERY: "How does the system detect emotions from text or images?"

  --- Result 1 (Relevance: 66.87%) ---
  Chunk ID:    98
  Text:        32 
 CONCLUSION AND FUTURE DIRECTIONS  
 
The distributed emotion detection system developed in this project demonstrates a robust and 
innovative approach to understanding human emotions through text. With an accuracy of 0.96, 
the system effectively identifies primary and secondary emotions, ackno...

  --- Result 2 (Relevance: 66.18%) ---
  Chunk ID:    90
  Text:        29 
 RESULTS AND DISCUSSION  
 
The distributed emotion detection system achieved an overall accuracy of 0.96, demonstrating its 
effectiveness in identifying both primary and secondary emotions from text inputs. This high 
accuracy underscores the potential of the system for applications requiring ...

  --- Result 3 (Relevance: 65.98%) ---
  Chunk ID:    40
  Text:        13 
 CHAPTER 1: MODEL TRAINING  
 
The first critical step in the development of the emotion detection syst

In [11]:
# =============================================================================
# CELL 7d: Remove front-matter boilerplate and re-ingest
# =============================================================================
# Your document has ~4-5 pages of academic boilerplate at the start:
#   title page, certificate, declaration, dedication, acknowledgements, TOC
#
# Strategy: Find where the actual content begins (Abstract or Introduction)
# and discard everything before it.

# --- Step 1: Find where real content starts ---
# We'll scan chunks for common section headers that signal actual content
content_start_markers = ["abstract", "introduction", "background", "chapter"]

start_index = 0
for i, chunk in enumerate(filtered_chunks):
    chunk_lower = chunk.lower()
    if any(marker in chunk_lower for marker in content_start_markers):
        start_index = i
        print(f"✓ Content starts at chunk {i}:")
        print(f"  \"{chunk[:150]}...\"\n")
        break

# --- Step 2: Show what we're removing ---
print(f"Removing {start_index} front-matter chunks:")
for i in range(start_index):
    preview = filtered_chunks[i][:100].replace('\n', ' ')
    print(f"  [{i}] \"{preview}...\"")

# --- Step 3: Slice and re-ingest ---
clean_chunks = filtered_chunks[start_index:]
print(f"\n  Before cleanup: {len(filtered_chunks)} chunks")
print(f"  After cleanup:  {len(clean_chunks)} chunks")
print(f"  Removed:        {start_index} chunks\n")

# Clear and rebuild ChromaDB collection
chroma_client.delete_collection(name=CHROMA_COLLECTION)
collection = chroma_client.create_collection(
    name=CHROMA_COLLECTION,
    metadata={"hnsw:space": "cosine"}
)

print("Embedding clean chunks...")
all_embeddings = embedding_model.encode(clean_chunks, show_progress_bar=True)

ids = [f"chunk_{i}" for i in range(len(clean_chunks))]
source_filename = Path(SAMPLE_DOC_PATH).name
metadatas = [
    {"source": source_filename, "chunk_index": i, "char_count": len(c)}
    for i, c in enumerate(clean_chunks)
]

BATCH_SIZE = 100
for start in range(0, len(clean_chunks), BATCH_SIZE):
    end = min(start + BATCH_SIZE, len(clean_chunks))
    collection.add(
        ids=ids[start:end],
        embeddings=all_embeddings[start:end].tolist(),
        documents=clean_chunks[start:end],
        metadatas=metadatas[start:end]
    )

print(f"\n✓ Re-ingestion complete — {collection.count()} clean chunks stored")

# --- Step 4: Quick verification with a vague query ---
print(f"\n{'=' * 60}")
print("VERIFICATION: Vague query should now return real content")
print(f"{'=' * 60}")

results = retrieve("What is this project about?", top_k=3)
for i, (doc, meta, dist) in enumerate(zip(
    results["documents"], results["metadatas"], results["distances"]
)):
    similarity = 1 - dist
    print(f"\n  --- Result {i+1} (Relevance: {similarity:.2%}) ---")
    print(f"  Chunk ID:    {meta['chunk_index']}")
    print(f"  Text:        {doc[:300]}{'...' if len(doc) > 300 else ''}")

✓ Content starts at chunk 8:
  "Pronaya Bhattacharya 
for his utmost valuable inputs and suggestions that have given my project broader horizons to 
aim at. ANGSHUMAN BASU  
A9101125..."

Removing 8 front-matter chunks:
  [0] "1   Distributed Emotion Detection System   Major Project submitted to   Amity School of Engineering ..."
  [1] "Pronaya Bhattacharya  (309881 )      Amity School of Engineering and Technology   Amity University  ..."
  [2] "The thesis is his original work completed after careful research and investigation. The  work of the..."
  [3] "Name and Signature of the   Head of Department                                   ……………………………... Name..."
  [4] "Angshuman Basu, hereby declare that the project report entitled  “Distributed Emotion Detection Syst..."
  [5] "Signature of Student 4   DEDICATION     This project report is dedicated to researchers, technologis..."
  [6] "Moreover, it is  dedicated to individuals and communities who advocate for the ethical  development .

Batches: 100%|██████████| 4/4 [00:00<00:00, 12.62it/s]


✓ Re-ingestion complete — 109 clean chunks stored

VERIFICATION: Vague query should now return real content

  --- Result 1 (Relevance: 28.57%) ---
  Chunk ID:    31
  Text:        This modular, transparent, and scalable 
approach is designed to overcome the limitations of existing systems and offer a more accurate, 
interpretable, and reliable solution for real -world emotion recognition applications. 12 
 METHODOLOGY  
 
 
 
Fig 1: The key processes involved in developing th...

  --- Result 2 (Relevance: 27.51%) ---
  Chunk ID:    0
  Text:        Pronaya Bhattacharya 
for his utmost valuable inputs and suggestions that have given my project broader horizons to 
aim at. ANGSHUMAN BASU  
A910112520006 6 
 ABSTRACT  
 
 
This project introduces a distributed framework for emotion detection that aims to enhance the 
scalability, reliability, and...

  --- Result 3 (Relevance: 26.29%) ---
  Chunk ID:    93
  Text:        Addressing these issues 
will unlock even greater potential for t

In [12]:
# =============================================================================
# CELL 8: Full RAG Pipeline — Retrieval + Augmented Generation
# =============================================================================
# This is the core loop that ties everything together:
#   1. User asks a question
#   2. Retrieve relevant chunks from ChromaDB
#   3. Build an augmented prompt: system instructions + context + question
#   4. Send to Ollama (Mistral) via /api/chat
#   5. Return the grounded response
#
# The key insight: the LLM never sees your full document. It only sees
# the top-k most relevant chunks, pasted into the prompt as "context."
# This keeps the prompt small and focused.

# --- System prompt ---
# This instructs the LLM on how to behave. It's critical for grounding —
# without "answer based ONLY on the provided context", the model will
# happily hallucinate from its training data.
SYSTEM_PROMPT = """You are a helpful assistant that answers questions based on the provided context.

RULES:
1. Answer the question using ONLY the information in the provided context.
2. If the context does not contain enough information to answer, say "I don't have enough information in the provided documents to answer that."
3. Be concise and direct.
4. If relevant, mention which part of the context your answer is based on.
5. Do not make up or infer information beyond what the context explicitly states."""


def rag_query(question: str, top_k: int = TOP_K) -> dict:
    """
    Full RAG pipeline: retrieve context, augment prompt, generate answer.
    
    Args:
        question: The user's question
        top_k:    Number of chunks to retrieve
        
    Returns:
        dict with:
            - answer:    The LLM's response text
            - context:   The formatted context that was sent to the LLM
            - sources:   List of chunk metadata for attribution
            - question:  The original question (echoed back)
    """
    # Step 1: Retrieve relevant chunks
    retrieved = retrieve(question, top_k=top_k)
    
    # Step 2: Format the context
    context = format_context(retrieved)
    
    # Step 3: Build the user message with context + question
    # This is the "augmented" part of RAG — we augment the user's
    # question with retrieved context before sending to the LLM.
    user_message = f"""Context:
{context}

---

Question: {question}"""
    
    # Step 4: Send to Ollama
    response = requests.post(
        f"{OLLAMA_BASE_URL}/api/chat",
        json={
            "model": OLLAMA_MODEL,
            "messages": [
                {"role": "system", "content": SYSTEM_PROMPT},
                {"role": "user", "content": user_message}
            ],
            "stream": False
        }
    )
    
    data = response.json()
    answer = data["message"]["content"]
    
    # Step 5: Package the result
    return {
        "answer": answer,
        "context": context,
        "sources": retrieved["metadatas"],
        "question": question
    }


# =============================================================================
# TEST: Ask questions and see the full RAG pipeline in action
# =============================================================================
test_questions = [
    "What is this project about?",
    "What deep learning model is used for emotion detection?",
    "How is Docker used in the system?",
    "What accuracy did the system achieve?",
    "How does the blockchain component work?",
]

for question in test_questions:
    print(f"{'=' * 60}")
    print(f"Q: {question}")
    print(f"{'=' * 60}")
    
    result = rag_query(question)
    
    print(f"\nA: {result['answer']}")
    
    # Show which chunks were used
    chunk_ids = [m['chunk_index'] for m in result['sources']]
    print(f"\n  Sources: chunks {chunk_ids}")
    print()

Q: What is this project about?

A:  Based on the provided context, this project is about developing a distributed framework for emotion detection. The goal is to enhance the scalability, reliability, and interpretability of traditional emotion recognition systems (Source: Chunk 0). The approach is modular, transparent, and scalable, and the framework aims to overcome the limitations of existing systems (Source: Chunk 31). The project also addresses ethical concerns by ensuring transparency and reliability through the use of blockchain technology (Source: Chunk 97). Future directions include modeling emotion relationships (Source: Chunk 93).

  Sources: chunks [31, 0, 93, 97, 58]

Q: What deep learning model is used for emotion detection?

A:  Based on the provided context, the deep learning models used for emotion detection are not explicitly stated. However, it is mentioned that many models leverage pre-trained transformers like BERT, RoBERTa, and GPT for text classification tasks (Ch